# Day 071 · 动量因子详解
**Momentum** · 阶段 P3 · 数据与因子工程

> 这一节讲一把跟前面完全不一样的尺子:动量因子。前面价值看便宜、成长看增速、质量看赚钱本事,都得翻财报。动量因子干脆不看基本面,只看一样东西——价格走势。它信奉四个字:强者恒强。说白了就是,过去一段时间一直在涨的票,赌它接下来继续涨;一直在跌的,赌它接着跌。听起来像追涨杀跌的赌徒逻辑,可它偏偏是全世界被验证最多、最稳健的因子之一,从美股到A股、从股票到大宗商品,几乎到处都有效。但动量有个最凶险的脾气:它平时赚得很稳,可一到市场剧烈转折的时候,会突然崩盘——之前跌最惨的输家会暴力反弹,把追强者的你一巴掌打回原形。这一节我们就把动量的赚钱逻辑和它的崩盘风险,一次讲透。

---

**课件生成日期:** 2026-06-27  ·  **建议学习时长:** 22 分钟

学习路径建议:1)先看视频建立直觉 → 2)阅读本 notebook 跑代码 → 3)看 PDF 课件复习要点 → 4)做自测题

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有需要的 Python 包,缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续下面的代码

> 这一格只在第一次跑要等几十秒,后面再开 notebook 就秒过。

In [1]:
# === 环境自检 + 自动安装(运行此单元格即可) ===
# 检测缺失的库 → 自动 pip 安装 → 注入中文字体 → 一行命令搞定
import importlib
import subprocess
import sys
import os

REQUIRED = ["baostock", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels", "yfinance"]
PIP_NAME = {
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "yaml": "PyYAML",
}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))

if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,正在自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置(让 matplotlib 不出乱码) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

CJK_FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",  # Linux/WSL
    "C:/Windows/Fonts/msyh.ttc",                               # Windows 微软雅黑
    "C:/Windows/Fonts/simhei.ttf",                             # Windows 黑体
    "/System/Library/Fonts/PingFang.ttc",                      # macOS 苹方
    "/System/Library/Fonts/STHeiti Medium.ttc",                # macOS 黑体
]
for p in CJK_FONT_PATHS:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft YaHei",
                                    "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪 — 现在可以跑下面的代码单元格")


✓ 所有 9 个必需库已就绪
✓ 中文字体已加载:NotoSansCJK-Regular.ttc
✓ 环境就绪 — 现在可以跑下面的代码单元格


## 学习目标

- 搞懂动量因子在赌什么:只看价格走势、不看基本面,赌「强者恒强、弱者恒弱」
- 理解动量为什么长期有效:背后是羊群效应和信息慢慢扩散,不是玄学
- 掌握经典的12-1动量:为什么是过去12个月、又为什么要剔除最近1个月
- 认清动量最凶险的脾气——动量崩盘:市场转折点上,输家暴力反弹会把动量策略打崩
- 理解动量和价值的低相关:两个因子常常一个灵一个不灵,组合起来更稳

## 历史背景:小李靠追最强的票,连着两年赚得盆满钵满,逢人就说自己悟了,结果市场一个急转弯,一个月把两年的超额全吐了回去

小李是个看图的短线高手,他不信什么财报,就一条:哪个板块最强、哪只票涨得最猛,他就上哪个。前两年市场结构性行情,强者恒强,小李这套追最强的打法如鱼得水,账户连着翻,他特别得意,逢人就说基本面都是骗人的、跟着趋势走才是王道。可天有不测风云。某一次,市场风格突然来了个180度急转弯,之前被资金抛弃、跌得最惨的那批低估值老票,因为太便宜、太多人不看好,突然被一波资金疯狂买回来,一个月暴力反弹。而小李重仓的那些前期最强的票,涨太多了、估值太高,反而被资金抛售,哗哗地跌。一来一回,小李一个月就把过去两年辛苦赚来的超额收益,几乎全吐了回去。他这才明白,动量这东西平时很灵,可一旦市场拐弯,它会瞬间反噬——追得越猛,崩得越狠。这就是动量最危险的地方:动量崩盘。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. 1. 动量因子:只看走势,赌强者恒强

动量因子是个异类:它完全不看公司基本面、不看便宜不便宜、不看赚不赚钱,只看一样——过去这段时间股价的走势。逻辑特别简单:过去涨得好的,赌它继续涨;过去跌得惨的,赌它接着跌。也就是「强者恒强、弱者恒弱」。它赚的不是公司变好的钱,而是趋势延续的钱、人性追涨杀跌的钱。这听起来不靠谱,却是金融史上被全球反复验证、最顽强的一个赚钱规律。


### 2. 2. 动量为什么有效:羊群 + 信息扩散慢

动量不是玄学,背后有两个扎实的原因。一是羊群效应:一只票开始涨,会吸引更多人跟风买,越买越涨,趋势自我强化。二是信息扩散慢:一家公司出了好消息,不是所有人同一秒知道,消息会一层层慢慢传开,股价也跟着一波波往上走,而不是一步到位。这两个人性和信息的特点,让「过去强的、未来一段时间还会强」这件事,在统计上长期成立。


### 3. 3. 经典的12-1动量:为什么剔除最近1月

学术界最经典的动量算法叫12-1。意思是:用过去12个月的累计涨幅来衡量一只票强不强,但要把最近这1个月剔除掉。为什么要剔除最近1个月?因为有个跟动量唱反调的短期反转效应——刚刚暴涨的票,接下来一两周常常会回吐一下。如果你把最近1个月也算进动量信号,就会被这个短期回吐污染,买在最高点。剔除掉最近1月,信号反而更干净、更稳。这是动量因子一个很重要的实操细节。


### 4. 4. 动量崩盘:最凶险的脾气

动量平时赚得稳,但它有个致命弱点叫动量崩盘。它往往发生在市场剧烈转折的时候:之前跌得最惨的输家,因为太便宜、太多人不看好,一旦情绪反转,会突然暴力反弹;而之前最强的赢家,涨太多估值太高,反而被抛售。这一来一回,「买赢家、卖输家」的动量策略会在很短时间里巨亏。历史上几次大的市场拐点,动量策略都吃过这种闷亏。所以玩动量,必须对崩盘有敬畏,有风控。


### 5. 5. 动量和价值低相关:天生一对

有意思的是,动量和价值这两个因子,常常一个灵的时候另一个不灵。价值买便宜的、被冷落的票,动量买正热的、最强的票,选股方向几乎相反,所以它们的收益相关性很低。这正是组合的黄金搭档:价值熄火的时候动量可能在赚,动量崩盘的时候价值可能在扛。两个一起用,整体的净值曲线会平稳很多,这也是很多量化策略同时配价值和动量的原因。


## 实操:动量因子:经典 12-1 动量分层回测 + 动量崩盘

下面这段代码跟视频里讲解的 highlights 是一致的,可以**直接 Run All** 看结果。

**依赖安装:**
```bash
pip install pandas numpy matplotlib yfinance akshare statsmodels
```


In [2]:
# day_071_momentum_factor.py — 动量因子:经典 12-1 动量分层回测 + 动量崩盘
# 真名上屏:baostock / query_history_k_data_plus(前复权 close) / 12-1 动量信号(过去12月剔除最近1月) /
#           月末调仓 / nlargest 赢家组(动量最高1/3) / nsmallest 输家组(动量最低1/3) / 多空净值最大回撤=动量崩盘
# 核心类比:动量=「强者恒强、弱者恒弱」,买过去一年涨最多的(赢家)、卖跌最多的(输家);
#           但为什么要剔除最近1个月?因为最近1个月有「短期反转」噪音(刚暴涨的下个月常回吐),
#           skip 掉它信号更干净。动量崩盘=市场急速反转时,跌最惨的输家暴力反弹反超赢家,
#           「买赢家卖输家」一夜被打回去——这是动量策略最致命的尾部风险。
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import baostock as bs


def _data_path(_name):
    # 铁律62:data/ 放在 notebook 文件夹里。仓库根(run_lab)存取 out/notebook/data/;
    # 原版 notebook 在 out/notebook/ 用 data/;中国版在 out/notebook/cn/ 用 ../data/
    from pathlib import Path as _P
    _here = _P.cwd()
    for _b in [_here / 'data', _here / '..' / 'data', _here / 'out' / 'notebook' / 'data', _here / '..' / '..' / 'data', _here / '..' / '..' / '..' / 'data']:
        if (_b / _name).exists():
            return str(_b / _name)
    if (_here / 'out' / 'notebook').exists():
        _t = _here / 'out' / 'notebook' / 'data'
    elif _here.name == 'cn':
        _t = _here / '..' / 'data'
    else:
        _t = _here / 'data'
    _t.mkdir(parents=True, exist_ok=True)
    return str(_t / _name)


pd.set_option('display.width', 160)
plt.rcParams['axes.unicode_minus'] = False

# ==== 标的池:12 只截面风格分化大的票(高弹性科技/AI/光模块 + 周期/资源大起大落)====
# 动量需要「有涨有跌、强弱分明」的截面:涨起来一年翻几倍、跌起来腰斩,赢家/输家才拉得开。
# 全部 2018 年初前已上市,保证整段回测从 2019 年(首个12月信号成型)起完整。
STOCKS = {
    # 高弹性 科技/AI/光模块(强者恒强的典型,2023-2024 AI 行情暴涨)
    '中际旭创': 'sz.300308', '沪电股份': 'sz.002463', '胜宏科技': 'sz.300476', '赛力斯': 'sh.601127',
    # 周期/资源(大起大落,2021 锂电/稀土暴涨、2022-2023 暴跌,动量信号反转剧烈)
    '北方稀土': 'sh.600111', '天齐锂业': 'sz.002466', '盐湖股份': 'sz.000792', '兖矿能源': 'sh.600188',
    '中远海控': 'sh.601919', '洛阳钼业': 'sh.603993', '华友钴业': 'sh.603799', '山东黄金': 'sh.600547',
}
START, END = '2018-01-01', '2024-12-31'
CSV_PX = _data_path('D071_momentum_close.csv')   # 前复权收盘(算策略真实收益 + 算动量信号)


# ==== 0. 拉数据:前复权收盘(adjustflag='2')。load 优先 + fetch 兜底 + to_csv 保存(铁律62)====
def _fetch():
    bs.login()
    px = {}
    for name, code in STOCKS.items():
        rs = bs.query_history_k_data_plus(code, 'date,close', start_date=START, end_date=END, frequency='d', adjustflag='2')
        r = []
        while rs.error_code == '0' and rs.next():
            r.append(rs.get_row_data())
        s = pd.DataFrame(r, columns=['date', 'close'])
        s['close'] = pd.to_numeric(s['close'], errors='coerce')
        s = s.set_index('date')
        px[name] = s['close']
    bs.logout()
    return pd.DataFrame(px)


if os.path.exists(CSV_PX):
    print('[数据] 从本地 CSV 读取 前复权收盘 ->', CSV_PX)
    px = pd.read_csv(CSV_PX, parse_dates=['date']).set_index('date')
else:
    print('[数据] baostock 拉取 12 只票 前复权收盘(adjustflag=2)...')
    px = _fetch()
    px.index.name = 'date'
    px.to_csv(CSV_PX)             # 拉完保存 CSV(铁律62)
    print('[数据] 已存 CSV ->', CSV_PX)

px.index = pd.to_datetime(px.index)   # 两条路径(CSV/fetch)统一成日期索引

print('\n==== 动量因子:经典 12-1 分层回测 + 动量崩盘 ====')
print('标的池 %d 只 · %s ~ %s' % (len(STOCKS), px.index[0].date(), px.index[-1].date()))


def maxdd(equity):
    # 最大回撤(%):从历史最高点跌下来最深的那一下
    e = pd.Series(equity).dropna()
    if len(e) < 2:
        return 0.0
    peak = e.cummax()
    return float((e / peak - 1.0).min() * 100)


def maxdd_window(equity):
    # 最大回撤的「区间」:返回(高点日期, 低点日期, 回撤%)——用来定位动量崩盘那一段
    e = pd.Series(equity).dropna()
    if len(e) < 2:
        return None, None, 0.0
    peak = e.cummax()
    dd = e / peak - 1.0
    trough = dd.idxmin()
    peak_d = e.loc[:trough].idxmax()
    return peak_d, trough, float(dd.min() * 100)


def ann_return(equity, n_months):
    # 年化收益(%):把总收益按月数折算成每年
    e = pd.Series(equity).dropna()
    if len(e) < 2 or n_months <= 0:
        return 0.0
    yrs = n_months / 12.0
    base = e.iloc[-1]
    if base <= 0:
        return float('nan')
    return float((base ** (1.0 / yrs) - 1.0) * 100)


# ==== 1. 月末重采样 + 两套动量信号 ====
monthly = px.resample('ME').last()                 # 月末收盘
mret = monthly.pct_change()                        # 本月相对上月的收益
fwd = mret.shift(-1)                               # 本月末建仓 -> 持有到下月末的收益(策略真实收益)

# 12-1 动量(skip 最近1月):信号 = 过去 t-12 到 t-1 个月的累计涨幅
#   = 上月末价 / 13个月前价 - 1  ——剔除「最近1个月」避开短期反转噪音
mom_skip = monthly.shift(1) / monthly.shift(13) - 1.0
# 12-0 动量(不 skip):信号 = 过去 12 个月含最近 1 月 = 本月末价 / 12个月前价 - 1
mom_full = monthly / monthly.shift(12) - 1.0


def run_strategy(signal):
    # 按 signal 每月分组:赢家组(动量最高1/3)/ 输家组(动量最低1/3)/ 基准全池等权
    # 同时算「买赢家卖输家」多空组合的月度收益,用来定位动量崩盘
    win_r, lose_r, base_r, ls_r, dates = [], [], [], [], []
    for i in range(len(monthly.index) - 1):
        t = monthly.index[i]
        s = signal.loc[t].dropna()
        f = fwd.loc[t]
        n = len(s)
        if n >= 2:
            k = max(2, n // 3)              # 每组取约 1/3,至少 2 只
            win = s.nlargest(k).index        # 赢家:过去动量最高
            lose = s.nsmallest(k).index      # 输家:过去动量最低
            wr = f[win].mean()
            lr = f[lose].mean()
            br = f[s.index].mean()
        else:
            wr = lr = br = f.mean()
        wr = 0.0 if pd.isna(wr) else wr
        lr = 0.0 if pd.isna(lr) else lr
        br = 0.0 if pd.isna(br) else br
        win_r.append(wr); lose_r.append(lr); base_r.append(br); ls_r.append(wr - lr)
        dates.append(monthly.index[i + 1])
    idx = pd.DatetimeIndex(dates)
    out = pd.DataFrame({'win': win_r, 'lose': lose_r, 'base': base_r, 'ls': ls_r}, index=idx)
    eq = {
        'win': (1 + out['win']).cumprod(),
        'lose': (1 + out['lose']).cumprod(),
        'base': (1 + out['base']).cumprod(),
        'ls': (1 + out['ls']).cumprod(),     # 多空净值:每月「买赢家(+wr)、卖输家(-lr)」
    }
    return out, eq


ret_skip, eq_skip = run_strategy(mom_skip)
ret_full, eq_full = run_strategy(mom_full)

# 只统计信号成型后(有有效动量、净值真正开始变动)的月份
valid = ret_skip[(ret_skip['win'] != 0) | (ret_skip['lose'] != 0)].index
start_d = valid[0]
eq_win = eq_skip['win'].loc[start_d:]
eq_lose = eq_skip['lose'].loc[start_d:]
eq_base = eq_skip['base'].loc[start_d:]
eq_ls = eq_skip['ls'].loc[start_d:]
# 重新归一到信号成型日 = 1(前面几个月没信号,净值恒为1)
eq_win = eq_win / eq_win.iloc[0]
eq_lose = eq_lose / eq_lose.iloc[0]
eq_base = eq_base / eq_base.iloc[0]
eq_ls = eq_ls / eq_ls.iloc[0]
n_months = len(eq_win) - 1

print('\n[12-1 动量分层 — 赢家组(动量最高1/3) / 输家组(动量最低1/3) / 基准等权,共 %d 个月]' % n_months)
for tag, eq in [('赢家组(高动量)', eq_win), ('输家组(低动量)', eq_lose), ('基准等权', eq_base)]:
    tot = (eq.iloc[-1] - 1) * 100
    print('%s  总收益 %+.1f%%   年化 %+.1f%%   最大回撤 %.1f%%' % (
        tag, tot, ann_return(eq.values, n_months), maxdd(eq.values)))

_prem = (eq_win.iloc[-1] - eq_lose.iloc[-1]) * 100
print('[动量溢价] 全期赢家组终值 %.2f vs 输家组终值 %.2f —— 赢家比输家多赚 %.0f 个百分点(强者恒强成立)'
      % (eq_win.iloc[-1], eq_lose.iloc[-1], _prem))

# ==== 2. 动量崩盘定位:多空组合(买赢家卖输家)净值的最大回撤区间 ====
peak_d, trough_d, ls_dd = maxdd_window(eq_ls)
# 找崩盘最剧烈的单月:多空月度收益最负的那个月
worst_m = ret_skip['ls'].loc[start_d:].idxmin()
worst_v = ret_skip['ls'].loc[worst_m] * 100
# 崩盘那一段:赢家组 vs 输家组 各自表现(看输家是否暴力反弹反超)
win_seg = (eq_win.loc[peak_d:trough_d].iloc[-1] / eq_win.loc[peak_d:trough_d].iloc[0] - 1) * 100
lose_seg = (eq_lose.loc[peak_d:trough_d].iloc[-1] / eq_lose.loc[peak_d:trough_d].iloc[0] - 1) * 100

print('\n[动量崩盘] 「买赢家卖输家」多空净值 最大回撤区间:%s ~ %s,回撤 %.1f%%'
      % (peak_d.date(), trough_d.date(), ls_dd))
print('  崩盘最剧烈单月:%s,多空当月收益 %+.1f%%' % (worst_m.strftime('%Y-%m'), worst_v))
print('  这一段里:赢家组 %+.1f%% vs 输家组 %+.1f%% —— 之前跌最惨的输家暴力反弹反超,动量策略一夜被打回'
      % (win_seg, lose_seg))

# ==== 3. skip 1月 vs 不skip:验证「剔除最近1月」的必要性 ====
eq_win_full = eq_full['win'].loc[start_d:]
eq_win_full = eq_win_full / eq_win_full.iloc[0]
tot_skip = (eq_win.iloc[-1] - 1) * 100
tot_full = (eq_win_full.iloc[-1] - 1) * 100
print('\n[skip 1月 vs 不skip — 赢家组净值对比]')
print('  12-1 动量(skip 最近1月)赢家组:总收益 %+.1f%%   年化 %+.1f%%   最大回撤 %.1f%%'
      % (tot_skip, ann_return(eq_win.values, n_months), maxdd(eq_win.values)))
print('  12-0 动量(含最近1月)  赢家组:总收益 %+.1f%%   年化 %+.1f%%   最大回撤 %.1f%%'
      % (tot_full, ann_return(eq_win_full.values, n_months), maxdd(eq_win_full.values)))
print('  -> skip 掉最近1月避开「短期反转」噪音(刚暴涨的下月常回吐),信号更干净、收益更稳')

# ==== 4. 三张图 ====
# 图1:赢家组 vs 输家组 vs 基准 三条净值,终值标注,标题点出动量溢价
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(eq_win.index, eq_win.values, lw=2, c='#c62828', label='赢家组(高动量)')
ax.plot(eq_lose.index, eq_lose.values, lw=2, c='#2e7d32', label='输家组(低动量)')
ax.plot(eq_base.index, eq_base.values, lw=1.6, c='#777', ls='--', label='基准等权')
ax.axhline(1.0, c='k', lw=.8)
for eq, c in [(eq_win, '#c62828'), (eq_lose, '#2e7d32'), (eq_base, '#777')]:
    ax.text(eq.index[-1], eq.iloc[-1], ' %.2f' % eq.iloc[-1], color=c, fontweight='bold', va='center')
ax.set_title('12-1 动量分层:赢家组(高动量)vs 输家组(低动量)vs 基准 —— 赢家比输家多赚 %.0f 个百分点(强者恒强)'
             % _prem)
ax.set_ylabel('净值(信号成型日=1)')
ax.legend(loc='upper left')
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig('chart_01.png', dpi=110)
plt.close()

# 图2:动量崩盘放大 —— 多空净值全程 + 高亮崩盘区间 + 标注回撤幅度
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(eq_ls.index, eq_ls.values, lw=1.8, c='#5a189a', label='多空净值(买赢家·卖输家)')
ax.axhline(1.0, c='k', lw=.8)
ax.axvspan(peak_d, trough_d, color='#c62828', alpha=.15)   # 高亮崩盘区间
ax.scatter([peak_d, trough_d], [eq_ls.loc[peak_d], eq_ls.loc[trough_d]], c='#c62828', zorder=5, s=40)
_mid = peak_d + (trough_d - peak_d) / 2
ax.annotate('动量崩盘\n%s~%s\n回撤 %.0f%%' % (peak_d.strftime('%Y-%m'), trough_d.strftime('%Y-%m'), ls_dd),
            xy=(trough_d, eq_ls.loc[trough_d]), xytext=(_mid, eq_ls.max() * 0.85),
            ha='center', color='#c62828', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#c62828'))
ax.set_title('动量崩盘:急速反转时输家暴力反弹反超,「买赢家卖输家」多空组合单段回撤 %.0f%%(崩盘最猛单月 %s %+.0f%%)'
             % (ls_dd, worst_m.strftime('%Y-%m'), worst_v))
ax.set_ylabel('多空净值(起点=1)')
ax.legend(loc='upper left')
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig('chart_02.png', dpi=110)
plt.close()

# 图3:12-1 动量(skip 1月)赢家组 vs 不skip(含最近1月)赢家组 净值对比 —— 说明 skip 的必要性
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(eq_win.index, eq_win.values, lw=2, c='#1565c0', label='12-1 动量(skip 最近1月)')
ax.plot(eq_win_full.index, eq_win_full.values, lw=2, c='#ef6c00', ls='--', label='12-0 动量(含最近1月)')
ax.axhline(1.0, c='k', lw=.8)
for eq, c in [(eq_win, '#1565c0'), (eq_win_full, '#ef6c00')]:
    ax.text(eq.index[-1], eq.iloc[-1], ' %.2f' % eq.iloc[-1], color=c, fontweight='bold', va='center')
ax.set_title('为什么要 skip 最近1月:剔除短期反转噪音后赢家组终值 %.2f vs 含最近1月 %.2f —— 信号更干净'
             % (eq_win.iloc[-1], eq_win_full.iloc[-1]))
ax.set_ylabel('赢家组净值(信号成型日=1)')
ax.legend(loc='upper left')
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig('chart_03.png', dpi=110)
plt.close()

print('\n[done] 12-1 动量分层回测 + 动量崩盘 + skip 必要性实证完成 —— 3 图已出')


[数据] 从本地 CSV 读取 前复权收盘 -> /mnt/d/huizi_ai_project/ai_course_video/out/notebook/data/D071_momentum_close.csv

==== 动量因子:经典 12-1 分层回测 + 动量崩盘 ====
标的池 12 只 · 2018-01-02 ~ 2024-12-31

[12-1 动量分层 — 赢家组(动量最高1/3) / 输家组(动量最低1/3) / 基准等权,共 82 个月]
赢家组(高动量)  总收益 +472.0%   年化 +29.1%   最大回撤 -34.7%
输家组(低动量)  总收益 +144.0%   年化 +13.9%   最大回撤 -32.5%
基准等权  总收益 +355.2%   年化 +24.8%   最大回撤 -32.1%
[动量溢价] 全期赢家组终值 5.72 vs 输家组终值 2.44 —— 赢家比输家多赚 328 个百分点(强者恒强成立)

[动量崩盘] 「买赢家卖输家」多空净值 最大回撤区间:2021-08-31 ~ 2023-10-31,回撤 -64.7%
  崩盘最剧烈单月:2020-11,多空当月收益 -25.6%
  这一段里:赢家组 -27.3% vs 输家组 +69.1% —— 之前跌最惨的输家暴力反弹反超,动量策略一夜被打回

[skip 1月 vs 不skip — 赢家组净值对比]
  12-1 动量(skip 最近1月)赢家组:总收益 +472.0%   年化 +29.1%   最大回撤 -34.7%
  12-0 动量(含最近1月)  赢家组:总收益 +437.8%   年化 +27.9%   最大回撤 -32.2%
  -> skip 掉最近1月避开「短期反转」噪音(刚暴涨的下月常回吐),信号更干净、收益更稳

[done] 12-1 动量分层回测 + 动量崩盘 + skip 必要性实证完成 —— 3 图已出


## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 动量溢价·强者恒强 | N/A | 2018到2024这7 年,用12-1动量(过去12个月剔除最近1月)选股,赢家组(最强的1/3)总收益+472%、终值5.72,大幅跑赢输家组(最弱的1/3)+144%、终值2.44,动量溢价+328个点,也跑赢基准+355%。过去一直在涨的票接下来继续跑赢过去一直在跌的,强者恒强在A股这7年成立。 |
| 动量崩盘·转折点反噬 | N/A | 动量最凶险的是崩盘。把买赢家、卖输家做成多空净值,它从2021年8月的高点一路崩到2023年10月,最大回撤-64.7%、几乎腰斩还多。崩盘那段里,之前最强的赢家组跌了27.3%,之前跌最惨的输家组反而暴力反弹+69.1%——风格一拐弯,输家反超赢家,追强者的人一夜被打回原形。 |
| 为什么剔除最近1月 | N/A | 经典12-1动量要把最近1个月剔除掉。实测:剔除最近1月的赢家组+472%,而不剔除(含最近1月)只有+437.8%。原因是有个跟动量唱反调的短期反转——刚暴涨的票下个月常回吐一下。把最近1月算进信号就会被污染、买在最高点;剔除掉,信号更干净、收益更稳。 |
| 动量与价值低相关 | N/A | 动量追最热、最强的票,价值买最冷、最便宜的票,选股方向几乎相反,所以两个因子收益相关性很低。价值熄火时动量可能在赚,动量崩盘时价值可能在扛。一起用,整体净值会平稳很多。这就是为什么很多量化策略同时配价值和动量,而不是只押一个。 |


## 常见坑

### ⚠ 01. 直接用最近1个月追涨

不剔除最近1个月,直接追刚刚暴涨的票,很容易买在短期最高点、吃一波回吐。经典动量一定要用过去12个月剔除最近1个月,避开短期反转的污染。

### ⚠ 02. 在崩盘前的高位满仓追强

市场越疯狂、强者涨得越离谱的时候,往往离动量崩盘越近。这时候还满仓追最强的票,一旦风格切换,输家暴力反弹、赢家崩塌,你会损失惨重。要警惕极端情绪。

### ⚠ 03. 玩动量不设风控

动量崩盘来得又快又猛,没有止损和仓位控制,一次崩盘能吃掉好几年的盈利。动量策略必须配合风控:控制单一仓位、设置止损、市场极端时降低暴露。

### ⚠ 04. 把题材爆炒当动量

真正的动量是有持续性的趋势,而很多妖股、题材股的暴涨是一两天的资金博弈,没有基本面支撑,涨得快崩得也快。把这种短命爆炒当成动量去追,极容易接最后一棒。

### ⚠ 05. 忽略动量股已经很贵

一直在涨的票,往往估值也被推得很高。光看趋势不看估值,容易在泡沫顶端接盘。把动量和价值、质量结合,既追强、又不接离谱的高价,会安全很多。

## 实战 SOP · 用动量因子选股 — 几条铁规矩

1. 先想清楚动量在赌什么:赌强者恒强、趋势延续,它只看价格走势、不看基本面,赚的是趋势和人性的钱。
2. 用经典12-1:过去12个月涨幅、剔除最近1个月,避开短期反转的污染,信号更干净。
3. 永远对动量崩盘有敬畏:市场剧烈转折时输家会暴力反弹,动量策略会巨亏,必须设风控、控仓位、留止损。
4. 别把短命爆炒当动量:真正的动量是有持续性的趋势,一两天的题材博弈不是,容易接最后一棒。
5. 和价值组合用:动量和价值低相关,一个灵时另一个常不灵,搭配起来净值更稳,别只押一个。

> 把这段打印贴在你电脑边,执行 1000 次它会回报你。

## 总结 · 你应该带走的

2. 动量因子=只看价格走势、不看基本面,赌强者恒强:过去一直涨的继续涨、一直跌的接着跌。
3. 为什么有效:羊群效应(越买越涨)+信息扩散慢(好消息一波波传开),让强者恒强长期统计上成立。
4. 经典12-1:用过去12个月涨幅、剔除最近1个月衡量强弱;剔除最近1月避开短期反转污染,信号更干净。
5. 动量溢价实测:2018-2024赢家组+472%大幅跑赢输家组+144%,动量溢价+328个点,强者恒强成立。
6. 最凶险的脾气是动量崩盘:多空净值2021-2023回撤-64.7%,那段赢家-27.3%、输家暴力反弹+69.1%,追强者一夜被打回。
7. 动量和价值低相关,一个灵时另一个常不灵,搭配用净值更稳;玩动量必须有风控,对崩盘有敬畏。

## 自测题

**Q1.** 动量因子跟价值、成长、质量最大的不同是什么?它为什么敢完全不看公司基本面、只看价格走势就能赚钱?

**Q2.** 经典的12-1动量为什么要把最近1个月剔除掉?如果不剔除会发生什么?这和短期反转有什么关系?

**Q3.** 什么是动量崩盘?它通常在什么时候发生、为什么会让「买赢家卖输家」的策略巨亏?你打算怎么防范?

把答案写下来,3 天后再回看。

## 下一节预告

**Day 072 · 反转因子** (Reversal)

这一节我们追强者恒强,也见识了动量崩盘的凶险。下一节我们把镜头拉到一个更短的时间尺度,讲一个跟动量正好唱反调的因子:反转因子。在一周这种短周期里,跌最惨的票反而常常会反弹一下。同一批票,中期看动量要买赢家,短期看反转却要买输家——这两个看似矛盾的策略怎么共存?A股的短期反转又到底灵不灵?下一节揭晓。

## 推荐阅读

- Narasimhan Jegadeesh, Sheridan Titman《Returns to Buying Winners and Selling Losers》(1993/JF) — 动量因子的开山之作,首次系统证明买赢家卖输家能赚超额,12-1 动量的源头
- Mark Carhart《On Persistence in Mutual Fund Performance》(1997/JF) — 把动量加进三因子、做成经典的四因子模型,动量从此成为学界标配因子
- Clifford Asness, Tobias Moskowitz, Lasse Pedersen《Value and Momentum Everywhere》(2013/JF) — 证明价值和动量在全球各类资产里普遍有效、且彼此低相关,组合的理论基石
- Kent Daniel, Tobias Moskowitz《Momentum Crashes》(2016/JFE) — 专门研究动量崩盘:为什么市场反转时动量会巨亏,玩动量必读的风险警示
- 杰西·利弗莫尔《股票大作手回忆录》(经典/多版本) — 一个世纪前的趋势交易大师用血泪经验讲透「顺势而为」与「趋势反转」,动量与崩盘的人性读本